## Mid-Term Load Forecast — Alt Forecast Models EDA (Weather Zones, 2022 Feb - 2025 Dec)

Companion to `02_mtlf_eda.ipynb` (Actual/Selected/Error only). This notebook explores the
**7 alternate ERCOT forecast-model columns** (`A3, A6, E, E1, E2, E3, M`) parsed alongside
`Actual`/`Selected`, intended as a load-prediction feature (ensemble mean) rather than the
single-model error diagnostic. Per-model error and ensemble spread/error are deferred for now.

**Source:** `01_data/1.2_raw_api/mid_term_load_forecast_models_20220201_20251201.parquet`
- Raw source: `01_data/1.1_raw_bulk/Load Forcast/Mid_Term_Load_Forecast_Metrics_*.xlsx`
- Parsed by: `02_scripts/2_parsers/parse_mid_term_load_forecast_models_parquet.py`

**Schema**
- 8 ERCOT weather zones + system total (Coast, East, FarWest, NCent, North, SCent, South, West, ERCOT)
- Per zone: `actual, selected, a3, a6, e, e1, e2, e3, m` (9 zones x 9 cols + datetime = 82 cols)

**Output:** `01_data/2_cleaned/load/forecast/models/` (ERCOT zone only, for now — weather-zone
breakdown, per-model error, and ensemble spread/error deferred until needed)
- ensemble mean, ERCOT only: `model_ensemble_features_ERCOT_*.csv`
- ERCOT-only hourly table (actual + all models): `Hourly_ERCOT_all_models_*.csv`


In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PARQUET = Path("../../01_data/1.2_raw_api/mid_term_load_forecast_models_20220201_20251201.parquet")
OUT_DIR = Path("../../01_data/2_cleaned/load/forecast")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
con = duckdb.connect()
# validate time range and column names

column_schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET}')
""").df()
print('=== Schema ===')
print(column_schema.to_string(index=False))

time_range = con.execute(f"""
    SELECT
        MIN(datetime) AS min_date
        ,MAX(datetime) AS max_date
    FROM read_parquet('{PARQUET}')
""").fetchone()
print('\n=== Time Range ===')
print(f'{time_range[0]} to {time_range[1]}')

=== Schema ===
     column_name  column_type null  key default extra
        datetime TIMESTAMP_NS  YES None    None  None
    actual_Coast       DOUBLE  YES None    None  None
  selected_Coast       DOUBLE  YES None    None  None
        a3_Coast       DOUBLE  YES None    None  None
        a6_Coast       DOUBLE  YES None    None  None
         e_Coast       DOUBLE  YES None    None  None
        e1_Coast       DOUBLE  YES None    None  None
        e2_Coast       DOUBLE  YES None    None  None
        e3_Coast       DOUBLE  YES None    None  None
         m_Coast       DOUBLE  YES None    None  None
     actual_East       DOUBLE  YES None    None  None
   selected_East       DOUBLE  YES None    None  None
         a3_East       DOUBLE  YES None    None  None
         a6_East       DOUBLE  YES None    None  None
          e_East       DOUBLE  YES None    None  None
         e1_East       DOUBLE  YES None    None  None
         e2_East       DOUBLE  YES None    None  None
         e3_E

In [7]:
# load data filter to only the ERCOT system-wide zone
df = pd.read_parquet(PARQUET)
df.head(3)

,datetime,actual_Coast,selected_Coast,a3_Coast,a6_Coast,e_Coast,e1_Coast,e2_Coast,e3_Coast,m_Coast,...,m_West,actual_ERCOT,selected_ERCOT,a3_ERCOT,a6_ERCOT,e_ERCOT,e1_ERCOT,e2_ERCOT,e3_ERCOT,m_ERCOT
0,2022-02-01 01:00:00,9810.580078,9466.780273,9828.533203,9997.777344,9503.190430,9444.459961,9576.030273,9466.780273,9497.615234,...,967.532776,35709.832031,35148.0,36439.0,35843.0,34619.0,34616.0,35045.0,35148.0,34857.0
1,2022-02-01 02:00:00,9549.062500,9225.679688,9636.767578,9715.292969,9295.599609,9249.690430,9359.809570,9225.679688,9282.694336,...,937.474731,34733.136719,34054.0,35833.0,35123.0,33584.0,33475.0,33999.0,34054.0,33778.0
2,2022-02-01 03:00:00,9508.433594,9106.030273,9547.450195,9625.110352,9202.790039,9183.849609,9264.230469,9106.030273,9189.225586,...,901.666504,34448.484375,33502.0,35684.0,35008.0,33069.0,32889.0,33496.0,33502.0,33239.0


In [ ]:
# EXPORT hourly ERCOT-only table (actual + selected + all 7 alt models)
ercot_cols = [c for c in df.columns if c.endswith('_ERCOT')]
df_ercot = df[['datetime'] + ercot_cols]
df_ercot.to_csv(OUT_DIR/'Hourly_ERCOT_all_models_20220201_20251201.csv', index=False)
df_ercot.head(3)

,datetime,actual_ERCOT,selected_ERCOT,a3_ERCOT,a6_ERCOT,e_ERCOT,e1_ERCOT,e2_ERCOT,e3_ERCOT,m_ERCOT
0,2022-02-01 01:00:00,35709.832031,35148.0,36439.0,35843.0,34619.0,34616.0,35045.0,35148.0,34857.0
1,2022-02-01 02:00:00,34733.136719,34054.0,35833.0,35123.0,33584.0,33475.0,33999.0,34054.0,33778.0
2,2022-02-01 03:00:00,34448.484375,33502.0,35684.0,35008.0,33069.0,32889.0,33496.0,33502.0,33239.0


In [ ]:
# -- Null count by month --
df_missing = df[df.isna().any(axis=1)].copy()
df_missing['month'] = df_missing['datetime'].dt.strftime('%Y-%m')
print('=== Missing rows by month ===')
df_missing.groupby('month')[ercot_cols].count().reset_index()

=== Missing rows by month (ERCOT cols only, for readability) ===


,month,actual_ERCOT,selected_ERCOT,a3_ERCOT,a6_ERCOT,e_ERCOT,e1_ERCOT,e2_ERCOT,e3_ERCOT,m_ERCOT
0,2022-03,23,23,23,23,23,23,23,23,23
1,2022-04,1,1,1,1,1,1,1,1,1
2,2022-05,0,0,0,0,0,0,0,0,0
3,2022-06,0,0,0,0,0,0,0,0,0
4,2023-02,0,0,0,0,0,0,0,0,0
5,2023-03,0,0,0,0,0,0,0,0,0
6,2023-04,0,0,0,0,0,0,0,0,0
7,2023-05,23,23,23,23,23,23,23,23,23
8,2023-06,1,1,1,1,1,1,1,1,1
9,2023-11,0,0,0,0,0,0,0,0,0


In [8]:
# -- duplicate datetimes --
n_dup = df['datetime'].duplicated().sum()
print(f'Duplicate datetime rows: {n_dup}')

Duplicate datetime rows: 103


In [ ]:
# plots for df_ercot
df_ercot 